# 3 · Planning for many futures

Optimise a plan on 50 sampled futures (expected cost and CVaR) and judge it on 1000 others, paired against the plan for the average. See [Lesson 3](../docs/lessons/03-planning-for-many-futures.md).

In [1]:
import matplotlib.pyplot as plt
import numpy as np

from fracsched.problem import small_problem
from fracsched.utilization_model import default_model

rng = np.random.default_rng(1)
problem = small_problem(0)  # AVR-04's small instance on the flight-hour clock
model = default_model()

In [2]:
from pmo.ga import GAConfig

from fracsched.deterministic_baseline import average_plan
from fracsched.robust_scenario_scheduling import RobustConfig, run_robust_ga
from fracsched.scenarios import cost_table, evaluate_in_chunks

average = average_plan(problem, GAConfig(pop_size=40, generations=60, seed=0)).schedule
u_in, _ = model.sample(rng, 50, 3, 26)
table = cost_table(problem, u_in)
plans = {"average": average}
for objective in ("mean", "cvar"):
    cfg = RobustConfig(objective=objective, pop_size=40, generations=60, seed=0)
    plans[objective] = run_robust_ga(problem, table, cfg, seeds=[average]).schedule
u_out, _ = model.sample(rng, 1000, 3, 26)
evs = dict(zip(plans, evaluate_in_chunks(problem, u_out, list(plans.values()))))
for name, ev in evs.items():
    d = ev.total - evs["average"].total
    print(f"{name:8s} mean {ev.total.mean():9,.0f}  paired change {d.mean():8,.0f} ± {d.std() / np.sqrt(d.size):,.0f}")

average  mean   410,955  paired change        0 ± 0
mean     mean   394,125  paired change  -16,830 ± 1,295
cvar     mean   413,906  paired change    2,951 ± 2,057
